## DONNEES ELECTORAL

Bloc 1 — Imports et configuration

In [1]:
import pandas as pd
import sqlite3
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger(__name__)

Bloc 2-Chemins et constantes

In [29]:
RAW_DIR       = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2\01_raw\electoral")
PROCESSED_DIR = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\electoral")
DB_PATH       = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2\03_database\electoral.db")
EXPORT_DIR    = PROCESSED_DIR / "par_election"

ELECTIONS_CIBLES = [
    "2014_euro_t1",
    "2019_euro_t1",
    "2024_euro_t1",
    "2017_legi_t1",
    "2022_legi_t1",
    "2024_legi_t1",
    "2017_pres_t1",
    "2022_pres_t1",
]

CODE_DEPT_RHONE = "69"

Bloc 3-Chargement des fichiers bruts

In [3]:
log.info("Chargement de general_results.csv...")
df_general = pd.read_csv(
    RAW_DIR / "general_results.csv",
    sep=";",
    dtype=str,
    low_memory=False,
    encoding="utf-8"
)
log.info(f"general_results : {len(df_general):,} lignes | {df_general.shape[1]} colonnes")

log.info("Chargement de candidats_results.csv...")
df_candidats = pd.read_csv(
    RAW_DIR / "candidats_results.csv",
    sep=";",
    encoding="utf-8",
    dtype=str,        # tout en string, code_departement contient "2A", "2B", etc.
    low_memory=False
)
log.info(f"candidats_results : {len(df_candidats):,} lignes | {df_candidats.shape[1]} colonnes")

2026-06-03 01:21:30 [INFO] Chargement de general_results.csv...
2026-06-03 01:21:42 [INFO] general_results : 3,162,440 lignes | 25 colonnes
2026-06-03 01:21:42 [INFO] Chargement de candidats_results.csv...
2026-06-03 01:25:10 [INFO] candidats_results : 27,524,743 lignes | 18 colonnes


Bloc 4-Filtrage Rhone + elections cibles

In [4]:
# general_results
masque_g = (
    (df_general["code_departement"] == CODE_DEPT_RHONE) &
    (df_general["id_election"].isin(ELECTIONS_CIBLES))
)
df_general_rhone = df_general[masque_g].copy()
log.info(f"general_results apres filtrage : {len(df_general_rhone):,} lignes")
log.info(df_general_rhone["id_election"].value_counts().sort_index().to_string())

# candidats_results
masque_c = (
    (df_candidats["code_departement"] == CODE_DEPT_RHONE) &
    (df_candidats["id_election"].isin(ELECTIONS_CIBLES))
)
df_candidats_rhone = df_candidats[masque_c].copy()
log.info(f"candidats_results apres filtrage : {len(df_candidats_rhone):,} lignes")
log.info(df_candidats_rhone["id_election"].value_counts().sort_index().to_string())

2026-06-03 01:25:24 [INFO] general_results apres filtrage : 10,426 lignes
2026-06-03 01:25:24 [INFO] id_election
2014_euro_t1    1255
2017_legi_t1    1287
2017_pres_t1    1287
2019_euro_t1    1291
2022_legi_t1    1317
2022_pres_t1    1317
2024_euro_t1    1336
2024_legi_t1    1336
2026-06-03 01:25:26 [INFO] candidats_results apres filtrage : 197,402 lignes
2026-06-03 01:25:26 [INFO] id_election
2014_euro_t1    28865
2017_legi_t1    19492
2017_pres_t1    14157
2019_euro_t1    43894
2022_legi_t1    14780
2022_pres_t1    15804
2024_euro_t1    50768
2024_legi_t1     9642


Bloc 5-Nettoyage de general_results

In [5]:
cols_entiers = ["inscrits", "abstentions", "votants", "blancs", "nuls", "exprimes"]
cols_ratios  = [
    "ratio_abstentions_inscrits", "ratio_votants_inscrits",
    "ratio_blancs_inscrits", "ratio_blancs_votants",
    "ratio_nuls_inscrits", "ratio_nuls_votants",
    "ratio_exprimes_inscrits", "ratio_exprimes_votants"
]

for col in cols_entiers:
    df_general_rhone[col] = pd.to_numeric(df_general_rhone[col], errors="coerce").astype("Int64")

for col in cols_ratios:
    df_general_rhone[col] = pd.to_numeric(df_general_rhone[col], errors="coerce")

# Europeennes 2014 : blancs non comptabilises separement avant la loi de 2015
nb_blancs_manquants = df_general_rhone.loc[
    df_general_rhone["id_election"] == "2014_euro_t1", "blancs"
].isna().sum()
log.info(f"2014_euro_t1 : {nb_blancs_manquants} lignes sans valeur blancs (normal, NaN conserve)")

# Suppression des bureaux sans inscrits
avant = len(df_general_rhone)
df_general_rhone = df_general_rhone[
    df_general_rhone["inscrits"].notna() & (df_general_rhone["inscrits"] > 0)
].copy()
log.info(f"Bureaux sans inscrits supprimes : {avant - len(df_general_rhone)}")

# Verification de coherence
incoherents = df_general_rhone[df_general_rhone["votants"] > df_general_rhone["inscrits"]]
if len(incoherents) > 0:
    log.warning(f"{len(incoherents)} lignes avec votants > inscrits")
    display(incoherents[["id_election", "code_commune", "code_bv", "inscrits", "votants"]])
else:
    log.info("Coherence votants/inscrits : OK")

# Normalisation des libelles
df_general_rhone["libelle_departement"] = df_general_rhone["libelle_departement"].str.strip()
df_general_rhone["libelle_commune"]     = df_general_rhone["libelle_commune"].str.strip()

# Colonnes derivees
df_general_rhone["annee"]              = df_general_rhone["id_election"].str[:4].astype("Int64")
df_general_rhone["type_election"]      = df_general_rhone["id_election"].str.extract(r"^\d{4}_([a-z]+)_")
df_general_rhone["tour"]               = df_general_rhone["id_election"].str.extract(r"_t(\d)$").astype("Int64")
df_general_rhone["taux_participation"] = (
    df_general_rhone["votants"] / df_general_rhone["inscrits"] * 100
).round(2)

log.info(f"Nettoyage general_results termine : {len(df_general_rhone):,} lignes propres")
df_general_rhone.head(3)

2026-06-03 01:25:34 [INFO] 2014_euro_t1 : 1255 lignes sans valeur blancs (normal, NaN conserve)
2026-06-03 01:25:34 [INFO] Bureaux sans inscrits supprimes : 0
2026-06-03 01:25:34 [INFO] Coherence votants/inscrits : OK
2026-06-03 01:25:34 [INFO] Nettoyage general_results termine : 10,426 lignes propres


,id_election,id_brut_miom,code_departement,libelle_departement,code_canton,libelle_canton,code_commune,libelle_commune,code_circonscription,libelle_circonscription,...,ratio_blancs_inscrits,ratio_blancs_votants,ratio_nuls_inscrits,ratio_nuls_votants,ratio_exprimes_inscrits,ratio_exprimes_votants,annee,type_election,tour,taux_participation
134674,2022_pres_t1,69001_0001,69,Rhône,NaN,NaN,69001,Affoux,08,8ème circonscription,...,1.27,1.56,0.63,0.78,79.11,97.66,2022,pres,1,81.01
134675,2022_pres_t1,69002_0001,69,Rhône,NaN,NaN,69002,Aigueperse,09,9ème circonscription,...,2.30,3.14,0.46,0.63,70.51,96.23,2022,pres,1,73.27
134676,2022_pres_t1,69003_0001,69,Rhône,NaN,NaN,69003,Albigny-Sur-Saône,05,5ème circonscription,...,1.10,1.37,0.60,0.75,78.88,97.89,2022,pres,1,80.58


Bloc 6-Nettoyage de candidats_results

In [6]:
# Numeriques
df_candidats_rhone["voix"] = pd.to_numeric(
    df_candidats_rhone["voix"], errors="coerce"
).astype("Int64")

df_candidats_rhone["ratio_voix_inscrits"] = pd.to_numeric(
    df_candidats_rhone["ratio_voix_inscrits"], errors="coerce"
)
df_candidats_rhone["ratio_voix_exprimes"] = pd.to_numeric(
    df_candidats_rhone["ratio_voix_exprimes"], errors="coerce"
)

# Nettoyage des chaines
cols_texte = [
    "nuance", "sexe", "nom", "prenom", "liste",
    "libelle_abrege_liste", "libelle_etendu_liste",
    "nom_tete_liste", "binome"
]
for col in cols_texte:
    df_candidats_rhone[col] = df_candidats_rhone[col].str.strip()

df_candidats_rhone["nom"] = df_candidats_rhone["nom"].str.upper()

# Colonnes derivees
df_candidats_rhone["annee"]         = df_candidats_rhone["id_election"].str[:4].astype("Int64")
df_candidats_rhone["type_election"] = df_candidats_rhone["id_election"].str.extract(r"^\d{4}_([a-z]+)_")
df_candidats_rhone["tour"]          = df_candidats_rhone["id_election"].str.extract(r"_t(\d)$").astype("Int64")

# Verification
negatifs = df_candidats_rhone[df_candidats_rhone["voix"] < 0]
if len(negatifs) > 0:
    log.warning(f"{len(negatifs)} lignes avec voix negatives")
else:
    log.info("Coherence voix : OK")

log.info(f"Nettoyage candidats_results termine : {len(df_candidats_rhone):,} lignes propres")
df_candidats_rhone.head(3)

2026-06-03 01:25:53 [INFO] Coherence voix : OK
2026-06-03 01:25:53 [INFO] Nettoyage candidats_results termine : 197,402 lignes propres


,id_election,id_brut_miom,code_departement,code_commune,code_bv,no_panneau,voix,ratio_voix_inscrits,ratio_voix_exprimes,nuance,...,nom,prenom,liste,libelle_abrege_liste,libelle_etendu_liste,nom_tete_liste,binome,annee,type_election,tour
295475,2022_pres_t1,69001_0001,69,69001,0001,1,0,0.00,0.00,NaN,...,ARTHAUD,Nathalie,NaN,NaN,NaN,NaN,NaN,2022,pres,1
295476,2022_pres_t1,69002_0001,69,69002,0001,1,1,0.46,0.65,NaN,...,ARTHAUD,Nathalie,NaN,NaN,NaN,NaN,NaN,2022,pres,1
295477,2022_pres_t1,69003_0001,69,69003,0001,1,2,0.20,0.25,NaN,...,ARTHAUD,Nathalie,NaN,NaN,NaN,NaN,NaN,2022,pres,1


Bloc 7-Mapping nuances vers partis politiques

In [7]:
MAPPING_NUANCES = {
    # Extreme gauche
    "EXG"  : ("Extreme gauche",                        "Extreme gauche"),
    "LEXG" : ("Extreme gauche",                        "Extreme gauche"),
    "DXG"  : ("Divers extreme gauche",                 "Extreme gauche"),

    # Communistes
    "COM"  : ("Parti Communiste Francais",             "Gauche radicale"),
    "LCOM" : ("Parti Communiste Francais",             "Gauche radicale"),

    # France Insoumise / Front de Gauche / NUPES
    "FI"   : ("La France Insoumise",                   "Gauche radicale"),
    "LFI"  : ("La France Insoumise",                   "Gauche radicale"),
    "LFG"  : ("Front de Gauche / NUPES",               "Gauche radicale"),
    "NUP"  : ("NUPES",                                 "Gauche radicale"),

    # Gauche / PS
    "DVG"  : ("Divers gauche",                         "Gauche"),
    "LDVG" : ("Divers gauche",                         "Gauche"),
    "UG"   : ("Union de la gauche",                    "Gauche"),
    "LUG"  : ("Union de la gauche",                    "Gauche"),
    "RDG"  : ("Rassemblement de la gauche",            "Gauche"),
    "SOC"  : ("Parti Socialiste",                      "Gauche"),
    "LSOC" : ("Parti Socialiste",                      "Gauche"),

    # Ecologistes
    "ECO"  : ("Europe Ecologie Les Verts",             "Ecologie"),
    "LECO" : ("Europe Ecologie Les Verts",             "Ecologie"),

    # Centre / Macronistes
    "ENS"  : ("Ensemble (Renaissance / LREM)",         "Centre"),
    "LENS" : ("Ensemble (Renaissance / LREM)",         "Centre"),
    "REM"  : ("La Republique En Marche",               "Centre"),
    "DSV"  : ("Divers centre",                         "Centre"),
    "MDM"  : ("Mouvement Democrate",                   "Centre"),
    "UDI"  : ("Union des Democrates et Independants",  "Centre"),

    # Divers
    "DIV"  : ("Divers",                                "Divers"),
    "LDIV" : ("Divers",                                "Divers"),

    # Droite / LR
    "DVD"  : ("Divers droite",                         "Droite"),
    "LDVD" : ("Divers droite",                         "Droite"),
    "DVC"  : ("Divers droite",                         "Droite"),
    "LR"   : ("Les Republicains",                      "Droite"),
    "LLR"  : ("Les Republicains",                      "Droite"),
    "UMP"  : ("Union pour un Mouvement Populaire",     "Droite"),
    "LUMP" : ("Union pour un Mouvement Populaire",     "Droite"),

    # Souverainistes
    "DLF"  : ("Debout la France",                      "Droite souverainiste"),
    "LDLF" : ("Debout la France",                      "Droite souverainiste"),

    # Rassemblement National / Front National
    "FN"   : ("Rassemblement National",                "Extreme droite"),
    "LFN"  : ("Rassemblement National",                "Extreme droite"),
    "RN"   : ("Rassemblement National",                "Extreme droite"),
    "LRN"  : ("Rassemblement National",                "Extreme droite"),

    # Reconquete
    "REC"  : ("Reconquete",                            "Extreme droite"),
    "LREC" : ("Reconquete",                            "Extreme droite"),

    # Extreme droite autre
    "EXD"  : ("Extreme droite",                        "Extreme droite"),
    "LEXD" : ("Extreme droite",                        "Extreme droite"),
    "UXD"  : ("Union de l extreme droite",             "Extreme droite"),
    "LVEC" : ("Extreme droite",                        "Extreme droite"),

    # Regionalistes
    "REG"  : ("Regionaliste",                          "Divers"),
    "LREG" : ("Regionaliste",                          "Divers"),

    # Divers centre-gauche / centre-droit non classes
    "LUC"  : ("Union du centre",                       "Centre"),
    "LVEC" : ("Divers extreme droite",                 "Extreme droite"),
}

# Application du mapping
df_candidats_rhone["parti_politique"] = df_candidats_rhone["nuance"].map(
    {k: v[0] for k, v in MAPPING_NUANCES.items()}
).fillna("Inconnu")

df_candidats_rhone["famille_politique"] = df_candidats_rhone["nuance"].map(
    {k: v[1] for k, v in MAPPING_NUANCES.items()}
).fillna("Inconnu")

# Verification
nuances_presentes   = set(df_candidats_rhone["nuance"].dropna().unique())
nuances_non_mappees = nuances_presentes - set(MAPPING_NUANCES.keys())
if nuances_non_mappees:
    log.warning(f"Nuances encore sans mapping : {sorted(nuances_non_mappees)}")
else:
    log.info("Toutes les nuances sont mappees.")

print(
    df_candidats_rhone[["nuance", "parti_politique", "famille_politique"]]
    .drop_duplicates()
    .sort_values("nuance")
    .to_string(index=False)
)

2026-06-03 01:26:04 [INFO] Toutes les nuances sont mappees.


nuance                      parti_politique    famille_politique
   COM            Parti Communiste Francais      Gauche radicale
   DIV                               Divers               Divers
   DLF                     Debout la France Droite souverainiste
   DSV                        Divers centre               Centre
   DVC                        Divers droite               Droite
   DVD                        Divers droite               Droite
   DVG                        Divers gauche               Gauche
   DXG                Divers extreme gauche       Extreme gauche
   ECO            Europe Ecologie Les Verts             Ecologie
   ENS        Ensemble (Renaissance / LREM)               Centre
   EXD                       Extreme droite       Extreme droite
   EXG                       Extreme gauche       Extreme gauche
    FI                  La France Insoumise      Gauche radicale
    FN               Rassemblement National       Extreme droite
  LCOM            Parti C

Bloc 8-Mapping nuances presidentielles par nom de candidat

In [8]:
MAPPING_CANDIDATS_PRES = {
    # 2017
    "ARTHAUD"         : ("EXG", "Lutte Ouvriere",                    "Extreme gauche"),
    "POUTOU"          : ("EXG", "Nouveau Parti Anticapitaliste",      "Extreme gauche"),
    "MÉLENCHON"       : ("FI",  "La France Insoumise",               "Gauche radicale"),
    "MELENCHON"       : ("FI",  "La France Insoumise",               "Gauche radicale"),
    "HAMON"           : ("SOC", "Parti Socialiste",                  "Gauche"),
    "MACRON"          : ("ENS", "En Marche / Renaissance",           "Centre"),
    "FILLON"          : ("LR",  "Les Republicains",                  "Droite"),
    "DUPONT-AIGNAN"   : ("DLF", "Debout la France",                  "Droite souverainiste"),
    "LEPEN"           : ("RN",  "Rassemblement National",            "Extreme droite"),
    "LE PEN"          : ("RN",  "Rassemblement National",            "Extreme droite"),
    "ASSELINEAU"      : ("EXD", "Union Populaire Republicaine",      "Droite souverainiste"),
    "CHEMINADE"       : ("DIV", "Solidarite et Progres",             "Divers"),
    "LASSALLE"        : ("DIV", "Resistons",                         "Divers"),
    # 2022
    "ROUSSEL"         : ("COM", "Parti Communiste Francais",         "Gauche radicale"),
    "JADOT"           : ("ECO", "Europe Ecologie Les Verts",         "Ecologie"),
    "HIDALGO"         : ("SOC", "Parti Socialiste",                  "Gauche"),
    "PECRESSE"        : ("LR",  "Les Republicains",                  "Droite"),
    "PÉCRESSE"        : ("LR",  "Les Republicains",                  "Droite"),
    "ZEMMOUR"         : ("EXD", "Reconquete",                        "Extreme droite"),
    "TAUBIRA"         : ("DVG", "Divers gauche",                     "Gauche"),
}

def completer_nuances_presidentielles(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    masque_pres = df["type_election"] == "pres"

    for nom_candidat, (nuance, parti, famille) in MAPPING_CANDIDATS_PRES.items():
        masque_nom = (
            masque_pres &
            df["nom"].str.upper().str.contains(nom_candidat, na=False)
        )
        # On ecrase toujours pour les presidentielles, NaN ou pas
        df.loc[masque_nom, "nuance"]           = nuance
        df.loc[masque_nom, "parti_politique"]  = parti
        df.loc[masque_nom, "famille_politique"] = famille

    encore_vides = df[masque_pres & df["nuance"].isna()]["nom"].value_counts()
    if len(encore_vides) > 0:
        log.warning(f"Candidats presidentiels sans nuance apres mapping :\n{encore_vides.to_string()}")
    else:
        log.info("Toutes les nuances presidentielles sont renseignees.")

    return df

df_candidats_rhone = completer_nuances_presidentielles(df_candidats_rhone)

print("Verification:")
print(
    df_candidats_rhone[df_candidats_rhone["type_election"] == "pres"]
    [["nom", "nuance", "parti_politique", "famille_politique"]]
    .drop_duplicates()
    .sort_values("nom")
    .to_string(index=False)
)

2026-06-03 01:26:15 [INFO] Toutes les nuances presidentielles sont renseignees.


Verification:
          nom nuance               parti_politique    famille_politique
      ARTHAUD    EXG                Lutte Ouvriere       Extreme gauche
   ASSELINEAU    EXD  Union Populaire Republicaine Droite souverainiste
    CHEMINADE    DIV         Solidarite et Progres               Divers
DUPONT-AIGNAN    DLF              Debout la France Droite souverainiste
       FILLON     LR              Les Republicains               Droite
        HAMON    SOC              Parti Socialiste               Gauche
      HIDALGO    SOC              Parti Socialiste               Gauche
        JADOT    ECO     Europe Ecologie Les Verts             Ecologie
     LASSALLE    DIV                     Resistons               Divers
       LE PEN     RN        Rassemblement National       Extreme droite
       MACRON    ENS       En Marche / Renaissance               Centre
    MÉLENCHON     FI           La France Insoumise      Gauche radicale
       POUTOU    EXG Nouveau Parti Anticapitaliste

Bloc 9-Mapping listes europeennes 2019

In [9]:
MAPPING_LISTES_2019 = {
    "LUTTE OUVRIÈRE"              : ("LEXG", "Lutte Ouvriere",                 "Extreme gauche"),
    "RÉVOLUTIONNAIRE"             : ("LEXG", "Extreme gauche",                 "Extreme gauche"),
    "LA FRANCE INSOUMISE"         : ("LFI",  "La France Insoumise",            "Gauche radicale"),
    "POUR L'EUROPE DES GENS"      : ("LFG",  "Front de Gauche / NUPES",        "Gauche radicale"),
    "EUROPE AU SERVICE PEUPLES"   : ("LFG",  "Front de Gauche / NUPES",        "Gauche radicale"),
    "ENVIE D'EUROPE"              : ("LSOC", "Parti Socialiste",               "Gauche"),
    "À VOIX ÉGALES"               : ("LDVG", "Divers gauche",                  "Gauche"),
    "PRENEZ LE POUVOIR"           : ("LDVG", "Divers gauche",                  "Gauche"),
    "LISTE CITOYENNE"             : ("LDVG", "Divers gauche",                  "Gauche"),
    "INITIATIVE CITOYENNE"        : ("LDVG", "Divers gauche",                  "Gauche"),
    "EUROPE ÉCOLOGIE"             : ("LECO", "Europe Ecologie Les Verts",      "Ecologie"),
    "URGENCE ÉCOLOGIE"            : ("LECO", "Europe Ecologie Les Verts",      "Ecologie"),
    "DÉCROISSANCE 2019"           : ("LECO", "Decroissance",                   "Ecologie"),
    "PARTI ANIMALISTE"            : ("LECO", "Parti Animaliste",               "Ecologie"),
    "RENAISSANCE"                 : ("LENS", "Ensemble (Renaissance / LREM)",  "Centre"),
    "LES EUROPÉENS"               : ("LENS", "Ensemble (Renaissance / LREM)",  "Centre"),
    "LUC"                         : ("LUC",  "Union du centre",                "Centre"),
    "PARTI FED. EUROPÉEN"         : ("LDIV", "Divers",                         "Divers"),
    "ÉVOLUTION CITOYENNE"         : ("LDIV", "Divers",                         "Divers"),
    "NEUTRE ET ACTIF"             : ("LDIV", "Divers",                         "Divers"),
    "DÉMOCRATIE REPRÉSENTATIVE"   : ("LDIV", "Divers",                         "Divers"),
    "ESPERANTO"                   : ("LDIV", "Divers",                         "Divers"),
    "PACE"                        : ("LDIV", "Divers",                         "Divers"),
    "ALLONS ENFANTS"              : ("LDIV", "Divers",                         "Divers"),
    "ALLIANCE JAUNE"              : ("LDIV", "Gilets Jaunes",                  "Divers"),
    "LES OUBLIES DE L'EUROPE"     : ("LDIV", "Divers",                         "Divers"),
    "PARTI PIRATE"                : ("LDIV", "Divers",                         "Divers"),
    "UNION DROITE-CENTRE"         : ("LDVD", "Divers droite",                  "Droite"),
    "UDLEF"                       : ("LDVD", "Divers droite",                  "Droite"),
    "DEBOUT LA FRANCE"            : ("LDLF", "Debout la France",               "Droite souverainiste"),
    "ENSEMBLE POUR LE FREXIT"     : ("LDLF", "Souverainiste",                  "Droite souverainiste"),
    "ENSEMBLE PATRIOTES"          : ("LDLF", "Patriotes",                      "Droite souverainiste"),
    "LISTE DE LA RECONQUÊTE"      : ("LREC", "Reconquete",                     "Extreme droite"),
    "UNE FRANCE ROYALE"           : ("LEXD", "Extreme droite",                 "Extreme droite"),
    "LA LIGNE CLAIRE"             : ("LEXD", "Extreme droite",                 "Extreme droite"),
}

def mapper_listes_2019(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    masque_2019 = df["id_election"] == "2019_euro_t1"
    for nom_liste, (nuance, parti, famille) in MAPPING_LISTES_2019.items():
        masque_liste = masque_2019 & (df["nom"] == nom_liste)
        df.loc[masque_liste, "nuance"]           = nuance
        df.loc[masque_liste, "parti_politique"]  = parti
        df.loc[masque_liste, "famille_politique"] = famille
    encore_vides = df[masque_2019 & df["nuance"].isna()]["nom"].value_counts()
    if len(encore_vides) > 0:
        log.warning(f"Listes 2019 sans mapping :\n{encore_vides.to_string()}")
    else:
        log.info("Toutes les listes 2019 sont mappees.")
    return df

df_candidats_rhone = mapper_listes_2019(df_candidats_rhone)

2026-06-03 01:26:19 [INFO] Toutes les listes 2019 sont mappees.


Bloc 10-Correction nom europeennes + nettoyage colonnes

In [10]:
# Remplir nom avec libelle_abrege_liste pour les europeennes sans nom individuel
masque_euro_sans_nom = (
    df_candidats_rhone["type_election"] == "euro"
) & (
    df_candidats_rhone["nom"].isna()
)
df_candidats_rhone.loc[masque_euro_sans_nom, "nom"] = (
    df_candidats_rhone.loc[masque_euro_sans_nom, "libelle_abrege_liste"]
)
log.info("Nom rempli avec libelle_abrege_liste pour les europeennes sans nom.")

# Verification
print("NaN nom par election :")
print(
    df_candidats_rhone.groupby("id_election")["nom"]
    .apply(lambda x: x.isna().sum())
    .to_string()
)

2026-06-03 01:26:24 [INFO] Nom rempli avec libelle_abrege_liste pour les europeennes sans nom.


NaN nom par election :
id_election
2014_euro_t1    0
2017_legi_t1    0
2017_pres_t1    0
2019_euro_t1    0
2022_legi_t1    0
2022_pres_t1    0
2024_euro_t1    0
2024_legi_t1    0


Bloc 11-Agregation general_results au niveau commune

In [11]:
cles_gen = [
    "id_election", "annee", "type_election", "tour",
    "code_departement", "libelle_departement",
    "code_commune", "libelle_commune"
]
cols_somme = ["inscrits", "abstentions", "votants", "nuls", "exprimes"]

df_general_communes = (
    df_general_rhone
    .groupby(cles_gen, dropna=False)[cols_somme]
    .sum(min_count=1)
    .reset_index()
)

df_general_communes["taux_participation"]     = (df_general_communes["votants"]     / df_general_communes["inscrits"] * 100).round(2)
df_general_communes["taux_abstention"]        = (df_general_communes["abstentions"] / df_general_communes["inscrits"] * 100).round(2)
df_general_communes["taux_nuls_inscrits"]     = (df_general_communes["nuls"]        / df_general_communes["inscrits"] * 100).round(2)
df_general_communes["taux_exprimes_inscrits"] = (df_general_communes["exprimes"]    / df_general_communes["inscrits"] * 100).round(2)

log.info(f"Agregation general par commune : {len(df_general_communes):,} lignes")
log.info(f"Communes uniques : {df_general_communes['code_commune'].nunique()}")
df_general_communes.head(3)

2026-06-03 01:26:29 [INFO] Agregation general par commune : 2,181 lignes
2026-06-03 01:26:29 [INFO] Communes uniques : 288


,id_election,annee,type_election,tour,code_departement,libelle_departement,code_commune,libelle_commune,inscrits,abstentions,votants,nuls,exprimes,taux_participation,taux_abstention,taux_nuls_inscrits,taux_exprimes_inscrits
0,2014_euro_t1,2014,euro,1,69,Rhône,69001,Affoux,251,122,129,5,124,51.39,48.61,1.99,49.4
1,2014_euro_t1,2014,euro,1,69,Rhône,69002,Aigueperse,220,127,93,12,81,42.27,57.73,5.45,36.82
2,2014_euro_t1,2014,euro,1,69,Rhône,69003,Albigny-Sur-Saône,1656,937,719,15,704,43.42,56.58,0.91,42.51


Bloc 12 — Nettoyage colonnes general

In [12]:
# Supprimer les colonnes inutiles pour le ML
COLS_SUPPRIMER_GENERAL = [
    "code_canton",
    "libelle_canton",
    "blancs",
    "ratio_blancs_inscrits",
    "ratio_blancs_votants",
    "taux_blancs_inscrits",
]

for col in COLS_SUPPRIMER_GENERAL:
    if col in df_general_rhone.columns:
        df_general_rhone = df_general_rhone.drop(columns=[col])
    if col in df_general_communes.columns:
        df_general_communes = df_general_communes.drop(columns=[col])

# Mettre None pour circonscription sur les europeennes uniquement
for df in [df_general_rhone, df_general_communes]:
    masque_euro = df["type_election"] == "euro"
    if "code_circonscription" in df.columns:
        df.loc[masque_euro, "code_circonscription"]    = None
        df.loc[masque_euro, "libelle_circonscription"] = None

# Imputer 0 pour les 5 bureaux avec 0 votants
for col in ["ratio_nuls_votants", "ratio_exprimes_votants"]:
    if col in df_general_rhone.columns:
        nb = df_general_rhone[col].isna().sum()
        df_general_rhone[col] = df_general_rhone[col].fillna(0)
        log.info(f"{col} : {nb} NaN imputes a 0")

log.info("Nettoyage colonnes general termine.")

print("NaN restants general_communes :")
m = df_general_communes.isna().sum()
print(m[m > 0].to_string() if m.any() else "  aucune")

2026-06-03 01:26:37 [INFO] ratio_nuls_votants : 5 NaN imputes a 0
2026-06-03 01:26:37 [INFO] ratio_exprimes_votants : 5 NaN imputes a 0
2026-06-03 01:26:37 [INFO] Nettoyage colonnes general termine.


NaN restants general_communes :
  aucune


Bloc 13 — Agregation candidats_results au niveau commune

In [13]:
cles_cand = [
    "id_election", "annee", "type_election", "tour",
    "code_departement", "code_commune",
    "no_panneau", "nuance", "parti_politique", "famille_politique",
    "nom", "libelle_abrege_liste",
]

df_candidats_communes = (
    df_candidats_rhone
    .groupby(cles_cand, dropna=False)["voix"]
    .sum(min_count=1)
    .reset_index()
)

# Jointure totaux pour recalcul des ratios
totaux = df_general_communes[["id_election", "code_commune", "inscrits", "exprimes"]].copy()
df_candidats_communes = df_candidats_communes.merge(
    totaux, on=["id_election", "code_commune"], how="left"
)

df_candidats_communes["ratio_voix_inscrits"] = (
    df_candidats_communes["voix"] / df_candidats_communes["inscrits"] * 100
).round(2)
df_candidats_communes["ratio_voix_exprimes"] = (
    df_candidats_communes["voix"] / df_candidats_communes["exprimes"] * 100
).round(2)

df_candidats_communes = df_candidats_communes.drop(columns=["inscrits", "exprimes"])

log.info(f"Agregation candidats par commune : {len(df_candidats_communes):,} lignes")
df_candidats_communes.head(3)

2026-06-03 01:26:42 [INFO] Agregation candidats par commune : 40,681 lignes


,id_election,annee,type_election,tour,code_departement,code_commune,no_panneau,nuance,parti_politique,famille_politique,nom,libelle_abrege_liste,voix,ratio_voix_inscrits,ratio_voix_exprimes
0,2014_euro_t1,2014,euro,1,69,69001,10,LUMP,Union pour un Mouvement Populaire,Droite,MUSELIER,NaN,30,11.95,24.19
1,2014_euro_t1,2014,euro,1,69,69001,11,LDVD,Divers droite,Droite,LESCURE,NaN,1,0.4,0.81
2,2014_euro_t1,2014,euro,1,69,69001,12,LDVG,Divers gauche,Gauche,COUTELIS,NaN,4,1.59,3.23


Bloc 14 — Colonnes finales uniformes

In [14]:
# Remplir nom depuis libelle_abrege_liste pour europeennes sans nom dans communes
masque_euro_sans_nom_com = (
    df_candidats_communes["type_election"] == "euro"
) & (
    df_candidats_communes["nom"].isna()
)
df_candidats_communes.loc[masque_euro_sans_nom_com, "nom"] = (
    df_candidats_communes.loc[masque_euro_sans_nom_com, "libelle_abrege_liste"]
)

# Mapping listes 2019 sur communes
df_candidats_communes = mapper_listes_2019(df_candidats_communes)

# Jointure libelle commune
libelles = (
    df_general_communes[["code_commune", "libelle_commune"]]
    .drop_duplicates()
    .copy()
)
df_candidats_communes = df_candidats_communes.merge(
    libelles, on="code_commune", how="left"
)

# Colonnes finales
COLS_FINALES = [
    "id_election",
    "annee",
    "type_election",
    "tour",
    "code_commune",
    "libelle_commune",
    "nom",
    "nuance",
    "parti_politique",
    "famille_politique",
    "voix",
    "ratio_voix_inscrits",
    "ratio_voix_exprimes",
]

df_export = df_candidats_communes[COLS_FINALES].copy()
df_export = df_export.sort_values(["id_election", "code_commune", "nom"]).reset_index(drop=True)

print("Colonnes finales :")
print(list(df_export.columns))

print("\nNaN restants :")
m = df_export.isna().sum()
print(m[m > 0].to_string() if m.any() else "  aucune")

print(f"\nLignes : {len(df_export):,}")
print("\nElections presentes :")
print(df_export["id_election"].value_counts().sort_index().to_string())

2026-06-03 01:26:47 [INFO] Toutes les listes 2019 sont mappees.


Colonnes finales :
['id_election', 'annee', 'type_election', 'tour', 'code_commune', 'libelle_commune', 'nom', 'nuance', 'parti_politique', 'famille_politique', 'voix', 'ratio_voix_inscrits', 'ratio_voix_exprimes']

NaN restants :
  aucune

Lignes : 42,344

Elections presentes :
id_election
2014_euro_t1     6877
2017_legi_t1     4189
2017_pres_t1     3201
2019_euro_t1     9452
2022_legi_t1     2985
2022_pres_t1     3336
2024_euro_t1    10526
2024_legi_t1     1778


Bloc 15 — Rapport qualite final

In [15]:
def rapport_qualite(df, label):
    print(f"\n{'='*60}")
    print(f"  Rapport qualite : {label}")
    print(f"{'='*60}")
    print(f"  Lignes    : {len(df):,}")
    print(f"  Colonnes  : {df.shape[1]}")
    manquants = df.isna().sum()
    manquants = manquants[manquants > 0]
    print(f"\n  Valeurs manquantes :")
    print(manquants.to_string() if len(manquants) > 0 else "  aucune")
    print(f"\n  Elections presentes :")
    print(df["id_election"].value_counts().sort_index().to_string())

rapport_qualite(df_general_rhone,    "general - bureaux de vote")
rapport_qualite(df_general_communes, "general - communes")
rapport_qualite(df_export,           "export final")


  Rapport qualite : general - bureaux de vote
  Lignes    : 10,426
  Colonnes  : 24

  Valeurs manquantes :
code_circonscription       5218
libelle_circonscription    5218

  Elections presentes :
id_election
2014_euro_t1    1255
2017_legi_t1    1287
2017_pres_t1    1287
2019_euro_t1    1291
2022_legi_t1    1317
2022_pres_t1    1317
2024_euro_t1    1336
2024_legi_t1    1336

  Rapport qualite : general - communes
  Lignes    : 2,181
  Colonnes  : 17

  Valeurs manquantes :
  aucune

  Elections presentes :
id_election
2014_euro_t1    288
2017_legi_t1    280
2017_pres_t1    280
2019_euro_t1    267
2022_legi_t1    267
2022_pres_t1    267
2024_euro_t1    266
2024_legi_t1    266

  Rapport qualite : export final
  Lignes    : 42,344
  Colonnes  : 13

  Valeurs manquantes :
  aucune

  Elections presentes :
id_election
2014_euro_t1     6877
2017_legi_t1     4189
2017_pres_t1     3201
2019_euro_t1     9452
2022_legi_t1     2985
2022_pres_t1     3336
2024_euro_t1    10526
2024_legi_t1     17

Bloc 16 — Sauvegarde CSV globaux

In [30]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

exports_globaux = {
    "rhone_general_bureaux.csv"    : df_general_rhone,
    "rhone_general_communes.csv"   : df_general_communes,
    "rhone_candidats_bureaux.csv"  : df_candidats_rhone,
    "rhone_candidats_communes.csv" : df_candidats_communes,
    "rhone_export_final.csv"       : df_export,
}

for nom_fichier, df in exports_globaux.items():
    chemin = PROCESSED_DIR / nom_fichier
    df.to_csv(chemin, index=False, sep=";", encoding="utf-8-sig")
    log.info(f"CSV sauvegarde : {chemin}  ({len(df):,} lignes)")

2026-06-03 01:39:22 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\electoral\rhone_general_bureaux.csv  (10,426 lignes)
2026-06-03 01:39:22 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\electoral\rhone_general_communes.csv  (2,181 lignes)
2026-06-03 01:39:24 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\electoral\rhone_candidats_bureaux.csv  (197,402 lignes)
2026-06-03 01:39:24 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\electoral\rhone_candidats_communes.csv  (42,344 lignes)
2026-06-03 01:39:25 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\electoral\rhone_export_final.csv  (42,344 lignes)


Bloc 17 — Export un fichier CSV par election

In [31]:
from pathlib import Path
PROCESSED_DIR = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_processed\electoral")
EXPORT_DIR    = PROCESSED_DIR / "par_election"
DB_PATH       = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2\03_database\electoral.db")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for id_election in sorted(df_export["id_election"].unique()):
    df_slice = df_export[df_export["id_election"] == id_election].copy()
    chemin = EXPORT_DIR / f"rhone_{id_election}.csv"
    df_slice.to_csv(chemin, index=False, sep=";", encoding="utf-8-sig")
    print(
        f"rhone_{id_election}.csv "
        f"({len(df_slice):,} lignes | "
        f"{df_slice['code_commune'].nunique()} communes | "
        f"{df_slice['nom'].nunique()} candidats/listes)"
    )

rhone_2014_euro_t1.csv (6,877 lignes | 288 communes | 23 candidats/listes)
rhone_2017_legi_t1.csv (4,189 lignes | 280 communes | 212 candidats/listes)
rhone_2017_pres_t1.csv (3,201 lignes | 280 communes | 11 candidats/listes)
rhone_2019_euro_t1.csv (9,452 lignes | 267 communes | 34 candidats/listes)
rhone_2022_legi_t1.csv (2,985 lignes | 267 communes | 159 candidats/listes)
rhone_2022_pres_t1.csv (3,336 lignes | 267 communes | 12 candidats/listes)
rhone_2024_euro_t1.csv (10,526 lignes | 266 communes | 38 candidats/listes)
rhone_2024_legi_t1.csv (1,778 lignes | 266 communes | 103 candidats/listes)


Bloc 18 — Sauvegarde SQLite

In [32]:
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
conn = sqlite3.connect(DB_PATH)

tables = {
    "general_bureaux"    : df_general_rhone,
    "general_communes"   : df_general_communes,
    "candidats_bureaux"  : df_candidats_rhone,
    "candidats_communes" : df_candidats_communes,
    "export_final"       : df_export,
}

for nom_table, df in tables.items():
    df_sql = df.copy()
    for col in df_sql.select_dtypes(include="Int64").columns:
        df_sql[col] = df_sql[col].astype(object).where(df_sql[col].notna(), None)
    df_sql.to_sql(nom_table, conn, if_exists="replace", index=False)
    log.info(f"Table '{nom_table}' : {len(df_sql):,} lignes inserees")

curseur = conn.cursor()
index_a_creer = [
    ("idx_gen_bv_elec",    "general_bureaux",    "id_election"),
    ("idx_gen_bv_com",     "general_bureaux",    "code_commune"),
    ("idx_gen_com_elec",   "general_communes",   "id_election"),
    ("idx_gen_com_com",    "general_communes",   "code_commune"),
    ("idx_cand_bv_elec",   "candidats_bureaux",  "id_election"),
    ("idx_cand_bv_com",    "candidats_bureaux",  "code_commune"),
    ("idx_cand_com_elec",  "candidats_communes", "id_election"),
    ("idx_cand_com_com",   "candidats_communes", "code_commune"),
    ("idx_export_elec",    "export_final",       "id_election"),
    ("idx_export_com",     "export_final",       "code_commune"),
]
for nom_index, table, colonne in index_a_creer:
    curseur.execute(f"CREATE INDEX IF NOT EXISTS {nom_index} ON {table} ({colonne})")

conn.commit()
conn.close()
log.info(f"Base SQLite finalisee : {DB_PATH}")

2026-06-03 01:39:31 [INFO] Table 'general_bureaux' : 10,426 lignes inserees
2026-06-03 01:39:31 [INFO] Table 'general_communes' : 2,181 lignes inserees
2026-06-03 01:39:33 [INFO] Table 'candidats_bureaux' : 197,402 lignes inserees
2026-06-03 01:39:33 [INFO] Table 'candidats_communes' : 42,344 lignes inserees
2026-06-03 01:39:33 [INFO] Table 'export_final' : 42,344 lignes inserees
2026-06-03 01:39:34 [INFO] Base SQLite finalisee : C:\Users\NIAMBELE Siata\Desktop\MSPR2\03_database\electoral.db


Bloc 19 — Verification finale

In [33]:
conn = sqlite3.connect(DB_PATH)

print("Tables dans la base :")
tables_presentes = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
)
for table in tables_presentes["name"]:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0, 0]
    print(f"  {table:<25} {n:>10,} lignes")

conn.close()

print(f"\nFichiers CSV par election :")
for f in sorted(EXPORT_DIR.glob("*.csv")):
    taille = f.stat().st_size / 1024
    print(f"  {f.name:<35} {taille:>8.1f} Ko")

print(f"\nFichiers CSV globaux :")
for f in sorted(PROCESSED_DIR.glob("*.csv")):
    taille = f.stat().st_size / 1024
    print(f"  {f.name:<35} {taille:>8.1f} Ko")

Tables dans la base :
  general_bureaux               10,426 lignes
  general_communes               2,181 lignes
  candidats_bureaux            197,402 lignes
  candidats_communes            42,344 lignes
  export_final                  42,344 lignes

Fichiers CSV par election :
  rhone_2014_euro_t1.csv                 622.5 Ko
  rhone_2017_legi_t1.csv                 403.3 Ko
  rhone_2017_pres_t1.csv                 327.8 Ko
  rhone_2019_euro_t1.csv                 970.3 Ko
  rhone_2022_legi_t1.csv                 290.6 Ko
  rhone_2022_pres_t1.csv                 338.4 Ko
  rhone_2024_euro_t1.csv                1104.5 Ko
  rhone_2024_legi_t1.csv                 177.2 Ko

Fichiers CSV globaux :


## DONNEES SOCIOECO

Bloc 1-Imports et configuration

In [35]:
import pandas as pd
import sqlite3
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger(__name__)

Bloc 2-Chemins et constantes

In [40]:
RAW_SOCIO    = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2\01_raw\socioeco")
RAW_SSMSI    = RAW_SOCIO / "ssmsi"
STAGING_DIR  = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\socioeco")
DB_PATH      = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2\03_database\socioeco.db")

CODE_DEPT_RHONE = "69"
ANNEES_CIBLES   = [2017, 2022, 2024]

Bloc 3-Chargement et traitement des fichiers securite (10 fichiers ssmsi)

 Les 10 fichiers de delinquance ont tous la meme structure :
Code commune | Nom commune | Annee | Nombre diffuse | Moyenne depart. | Taux ...
Les NA sur "Nombre diffuse" sont remplaces par 0 comme convenu
Les 10 types de delinquance sont additionnes par commune et par annee

In [44]:
FICHIERS_SECURITE = {
    "cambriolages"             : RAW_SSMSI / "ssmsi_cambriolages.csv",
    "destructions"             : RAW_SSMSI / "ssmsi_destructions.csv",
    "escroqueries"             : RAW_SSMSI / "ssmsi_escroqueries.csv",
    "trafic_stupefiants"       : RAW_SSMSI / "ssmsi_trafic_stupefiants.csv",
    "usage_stupefiants"        : RAW_SSMSI / "ssmsi_usage_stupefiants.csv",
    "violences_hors_famille"   : RAW_SSMSI / "ssmsi_violences_hors_famille.csv",
    "violences_intrafamiliales": RAW_SSMSI / "ssmsi_violences_intrafamiliales.csv",
    "violences_sexuelles"      : RAW_SSMSI / "ssmsi_violences_sexuelles.csv",
    "vols_accessoires"         : RAW_SSMSI / "ssmsi_vols_accessoires.csv",
    "vols_avec_armes"          : RAW_SSMSI / "ssmsi_vols_avec_armes.csv",
    "vols_dans_vehicules"      : RAW_SSMSI / "ssmsi_vols_dans_vehicules.csv",
    "vols_sans_armes"          : RAW_SSMSI / "ssmsi_vols_sans_armes.csv",
    "vols_sans_violence"       : RAW_SSMSI / "ssmsi_vols_sans_violence.csv",
    "vols_vehicule"            : RAW_SSMSI / "ssmsi_vols_vehicule.csv",
}
morceaux = []

for type_delit, chemin in FICHIERS_SECURITE.items():
    df = pd.read_csv(chemin, dtype={"Code commune": str})

    # Filtrage Rhone et annees cibles
    masque = (
        df["Code commune"].str.startswith(CODE_DEPT_RHONE) &
        df["Année"].isin(ANNEES_CIBLES)
    )
    df = df[masque].copy()

    # Remplacement des NA par 0
    nb_na = df["Nombre diffusé"].isna().sum()
    df["Nombre diffusé"] = df["Nombre diffusé"].fillna(0)

    df = df.rename(columns={
        "Code commune"  : "code_commune",
        "Nom commune"   : "libelle_commune",
        "Année"         : "annee",
        "Nombre diffusé": "nombre",
    })[["code_commune", "libelle_commune", "annee", "nombre"]]

    df["type_delit"] = type_delit
    morceaux.append(df)

    log.info(f"{type_delit:<30} {len(df):>5} lignes | {nb_na} NA remplaces par 0")

df_securite_long = pd.concat(morceaux, ignore_index=True)

# Addition de tous les types de delinquance par commune et par annee
df_securite = (
    df_securite_long
    .groupby(["code_commune", "libelle_commune", "annee"])["nombre"]
    .sum()
    .reset_index()
    .rename(columns={"nombre": "total_delits"})
)

df_securite["annee"] = df_securite["annee"].astype(int)

log.info(f"Table securite finale : {len(df_securite):,} lignes")
log.info(f"Communes uniques : {df_securite['code_commune'].nunique()}")
log.info(f"Annees presentes : {sorted(df_securite['annee'].unique())}")

df_securite.head(6)

2026-06-03 15:35:01 [INFO] cambriolages                     825 lignes | 411 NA remplaces par 0
2026-06-03 15:35:02 [INFO] destructions                     825 lignes | 381 NA remplaces par 0
2026-06-03 15:35:02 [INFO] escroqueries                     825 lignes | 367 NA remplaces par 0
2026-06-03 15:35:03 [INFO] trafic_stupefiants               825 lignes | 332 NA remplaces par 0
2026-06-03 15:35:04 [INFO] usage_stupefiants                825 lignes | 470 NA remplaces par 0
2026-06-03 15:35:04 [INFO] violences_hors_famille           825 lignes | 462 NA remplaces par 0
2026-06-03 15:35:05 [INFO] violences_intrafamiliales        825 lignes | 530 NA remplaces par 0
2026-06-03 15:35:05 [INFO] violences_sexuelles              825 lignes | 578 NA remplaces par 0
2026-06-03 15:35:06 [INFO] vols_accessoires                 825 lignes | 441 NA remplaces par 0
2026-06-03 15:35:06 [INFO] vols_avec_armes                  825 lignes | 212 NA remplaces par 0
2026-06-03 15:35:07 [INFO] vols_dans_veh

,code_commune,libelle_commune,annee,total_delits
0,69001,Affoux,2017,0.0
1,69001,Affoux,2022,0.0
2,69001,Affoux,2024,0.0
3,69002,Aigueperse,2017,0.0
4,69002,Aigueperse,2022,0.0
5,69002,Aigueperse,2024,0.0


Bloc 4 — Population (population.CSV)

Le fichier contient plusieurs millesimes dans des colonnes separees
On extrait P22_POP (2022) et on utilisera P16_POP comme proxy 2017
car P17_POP n'existe pas dans ce fichier (recensement tous les 5 ans)

In [45]:
df_pop_raw = pd.read_csv(
    RAW_SOCIO / "population.CSV",
    sep=";",
    dtype={"CODGEO": str},
    low_memory=False
)

# Filtrage Rhone
df_pop_raw = df_pop_raw[df_pop_raw["CODGEO"].str.startswith(CODE_DEPT_RHONE)].copy()

# On garde les colonnes utiles : population et logements
cols_pop = {
    "CODGEO"    : "code_commune",
    "P22_POP"   : "population_2022",
    "P16_POP"   : "population_2016",
    "P22_LOG"   : "nb_logements_2022",
    "P16_LOG"   : "nb_logements_2016",
    "P22_RP"    : "nb_residences_principales_2022",
    "P16_RP"    : "nb_residences_principales_2016",
    "P22_LOGVAC": "nb_logements_vacants_2022",
    "P16_LOGVAC": "nb_logements_vacants_2016",
    "SUPERF"    : "superficie_km2",
}
cols_presentes = {k: v for k, v in cols_pop.items() if k in df_pop_raw.columns}
df_pop = df_pop_raw[list(cols_presentes.keys())].rename(columns=cols_presentes).copy()

# Conversion numerique
for col in df_pop.columns[1:]:
    df_pop[col] = pd.to_numeric(df_pop[col], errors="coerce")

# Densite de population
ddf_pop["densite_2022"] = (df_pop["population_2022"] / df_pop["superficie_km2"]).round(2)
df_pop["densite_2016"] = (df_pop["population_2016"] / df_pop["superficie_km2"]).round(2)

log.info(f"Table population : {len(df_pop):,} communes")
log.info(f"NaN : {df_pop.isna().sum()[df_pop.isna().sum() > 0].to_dict()}")

df_pop.head(3)

2026-06-03 15:36:39 [INFO] Table population : 275 communes
2026-06-03 15:36:39 [INFO] NaN : {}


,code_commune,population_2022,population_2017_proxy,nb_logements_2022,nb_residences_principales_2022,nb_logements_vacants_2022,superficie_km2,densite_2022
26941,69001,397,358,181.394436,153.300141,12.486353,10.65,37.28
26942,69002,239,251,196.061188,118.394522,35.558233,12.95,18.46
26943,69003,3013,2833,1247.256052,1124.664886,99.094526,2.57,1172.37


Bloc 5-Emploi et chomage (base-cc-emploi-pop-active-2022.CSV)

In [46]:
df_emploi_raw = pd.read_csv(
    RAW_SOCIO / "base-cc-emploi-pop-active-2022.CSV",
    sep=";",
    dtype={"CODGEO": str},
    low_memory=False
)

# Filtrage Rhone
df_emploi_raw = df_emploi_raw[df_emploi_raw["CODGEO"].str.startswith(CODE_DEPT_RHONE)].copy()

# Colonnes utiles pour le ML
# P22_ACT1564  = actifs 15-64 ans
# P22_CHOM1564 = chomeurs 15-64 ans
# P22_ACTOCC1564 = actifs occupes 15-64 ans
cols_emploi = {
    "CODGEO"          : "code_commune",
    "P22_ACT1564"     : "actifs_2022",
    "P22_CHOM1564"    : "chomeurs_2022",
    "P22_ACTOCC1564"  : "actifs_occupes_2022",
    "P22_INACT1564"   : "inactifs_2022",
}

cols_presentes = {k: v for k, v in cols_emploi.items() if k in df_emploi_raw.columns}
df_emploi = df_emploi_raw[list(cols_presentes.keys())].rename(columns=cols_presentes).copy()

for col in df_emploi.columns[1:]:
    df_emploi[col] = pd.to_numeric(df_emploi[col], errors="coerce")

# Taux de chomage calcule
df_emploi["taux_chomage_2022"] = (
    df_emploi["chomeurs_2022"] / df_emploi["actifs_2022"] * 100
).round(2)

log.info(f"Table emploi : {len(df_emploi):,} communes")
log.info(f"NaN : {df_emploi.isna().sum()[df_emploi.isna().sum() > 0].to_dict()}")

df_emploi.head(3)

2026-06-03 15:37:35 [INFO] Table emploi : 275 communes
2026-06-03 15:37:35 [INFO] NaN : {}


,code_commune,actifs_2022,chomeurs_2022,actifs_occupes_2022,inactifs_2022,taux_chomage_2022
26941,69001,220.265899,5.961562,214.304337,33.697223,2.71
26942,69002,105.061320,3.134262,101.927058,25.713797,2.98
26943,69003,1301.170369,130.353647,1170.816722,404.220484,10.02


Bloc 6-Revenus et pauvrete 2017 (cc_filosofi_2017_COM.CSV)

In [48]:
df_filo17_raw = pd.read_csv(
    RAW_SOCIO / "cc_filosofi_2017_COM.CSV",
    sep=";",
    dtype={"CODGEO": str},
    low_memory=False
)

# Filtrage Rhone
df_filo17_raw = df_filo17_raw[df_filo17_raw["CODGEO"].str.startswith(CODE_DEPT_RHONE)].copy()

cols_filo17 = {
    "CODGEO"  : "code_commune",
    "MED17"   : "revenu_median_2017",
    "TP6017"  : "taux_pauvrete_2017",
    "PCHO17"  : "part_chomeurs_2017",
    "PIMP17"  : "part_foyers_imposes_2017",
    "D117"    : "d1_revenu_2017",
    "D917"    : "d9_revenu_2017",
    "RD17"    : "ratio_interdecile_2017",
}

cols_presentes = {k: v for k, v in cols_filo17.items() if k in df_filo17_raw.columns}
df_filo17 = df_filo17_raw[list(cols_presentes.keys())].rename(columns=cols_presentes).copy()

for col in df_filo17.columns[1:]:
    df_filo17[col] = pd.to_numeric(df_filo17[col], errors="coerce")

# Imputation par la mediane departementale pour les communes sous le seuil de diffusion INSEE
# Ces communes sont generalement rurales avec moins de 1000 habitants
cols_a_imputer = [c for c in df_filo17.columns if c != "code_commune"]

log.info("Imputation par mediane departementale (secret statistique INSEE) :")
for col in cols_a_imputer:
    nb_nan  = df_filo17[col].isna().sum()
    if nb_nan > 0:
        mediane = df_filo17[col].median()
        df_filo17[col] = df_filo17[col].fillna(mediane)
        log.info(f"  {col:<35} {nb_nan:>3} NaN imputes | mediane = {mediane:.2f}")

# Verification
print("NaN restants apres imputation :")
m = df_filo17.isna().sum()
print(m[m > 0].to_string() if m.any() else "  aucune")

log.info(f"Table filosofi 2017 : {len(df_filo17):,} communes")
df_filo17.head(3)

2026-06-03 15:41:01 [INFO] Imputation par mediane departementale (secret statistique INSEE) :
2026-06-03 15:41:01 [INFO]   revenu_median_2017                    1 NaN imputes | mediane = 23390.00
2026-06-03 15:41:01 [INFO]   taux_pauvrete_2017                  188 NaN imputes | mediane = 9.45
2026-06-03 15:41:01 [INFO]   part_chomeurs_2017                  144 NaN imputes | mediane = 2.70
2026-06-03 15:41:01 [INFO]   part_foyers_imposes_2017            144 NaN imputes | mediane = 65.00
2026-06-03 15:41:01 [INFO]   d1_revenu_2017                      144 NaN imputes | mediane = 14235.00
2026-06-03 15:41:01 [INFO]   d9_revenu_2017                      144 NaN imputes | mediane = 41635.00
2026-06-03 15:41:01 [INFO]   ratio_interdecile_2017              144 NaN imputes | mediane = 3.00
2026-06-03 15:41:01 [INFO] Table filosofi 2017 : 276 communes


NaN restants apres imputation :
  aucune


,code_commune,revenu_median_2017,taux_pauvrete_2017,part_chomeurs_2017,part_foyers_imposes_2017,d1_revenu_2017,d9_revenu_2017,ratio_interdecile_2017
27000,69001,22610.0,9.45,2.7,65.0,14235.0,41635.0,3.0
27001,69002,19550.0,9.45,2.7,65.0,14235.0,41635.0,3.0
27002,69003,23390.0,10.00,3.7,61.0,12500.0,45230.0,3.6


Bloc 7-Revenus et pauvrete 2021 (FILO2021_DISP_COM.xlsx)

In [ ]:
# L'en-tete avec les codes courts est a la ligne 5 (index 5)
# Les donnees commencent a la ligne 6 (index 6)
df_filo21_raw = pd.read_excel(
    RAW_SOCIO / "FILO2021_DISP_COM.xlsx",
    sheet_name="ENSEMBLE",
    engine="calamine",
    header=5,
    dtype=str
)

# Filtrage Rhone
df_filo21_raw = df_filo21_raw[
    df_filo21_raw["CODGEO"].astype(str).str.startswith(CODE_DEPT_RHONE)
].copy()

cols_filo21 = {
    "CODGEO" : "code_commune",
    "Q221"   : "revenu_median_2021",
    "NBMEN21": "nb_menages_2021",
    "NBPERS21": "nb_personnes_menages_2021",
}

cols_presentes = {k: v for k, v in cols_filo21.items() if k in df_filo21_raw.columns}
df_filo21 = df_filo21_raw[list(cols_presentes.keys())].rename(columns=cols_presentes).copy()

# Conversion numerique : remplacer virgules et valeurs secretes 's' par NaN
for col in df_filo21.columns[1:]:
    df_filo21[col] = pd.to_numeric(
        df_filo21[col].astype(str).str.replace(",", ".").str.strip().replace("s", float("nan")),
        errors="coerce"
    )

# Imputation par la mediane departementale
log.info("Imputation par mediane departementale FILO 2021 :")
for col in df_filo21.columns[1:]:
    nb_nan = df_filo21[col].isna().sum()
    if nb_nan > 0:
        mediane = df_filo21[col].median()
        df_filo21[col] = df_filo21[col].fillna(mediane)
        log.info(f"  {col:<35} {nb_nan:>3} NaN imputes | mediane = {mediane:.2f}")

# Verification
print("NaN restants apres imputation :")
m = df_filo21.isna().sum()
print(m[m > 0].to_string() if m.any() else "  aucune")

log.info(f"Table filosofi 2021 : {len(df_filo21):,} communes")
df_filo21.head(3)

2026-06-03 15:43:03 [INFO] Imputation par mediane departementale FILO 2021 :
2026-06-03 15:43:03 [INFO]   revenu_median_2021                    1 NaN imputes | mediane = 25650.00
2026-06-03 15:43:03 [INFO]   nb_menages_2021                       1 NaN imputes | mediane = 798.00
2026-06-03 15:43:03 [INFO]   nb_personnes_menages_2021             1 NaN imputes | mediane = 1898.00
2026-06-03 15:43:03 [INFO] Table filosofi 2021 : 276 communes


NaN restants apres imputation :
  aucune


,code_commune,revenu_median_2021,nb_menages_2021,nb_personnes_menages_2021
26999,69001,24970.0,138.0,349.0
27000,69002,21290.0,112.0,227.0
27001,69003,27080.0,1085.0,2568.0


Bloc 8-Immigration 2017 (BTT_TD_IMG1A_2017.CSV)

IMMI = 1 : immigres | IMMI = 2 : non immigres
On somme toutes les tranches d age et les deux sexes par commune

In [55]:
df_imm17_raw = pd.read_csv(
    RAW_SOCIO / "BTT_TD_IMG1A_2017.CSV",
    sep=";",
    dtype=str,
    low_memory=False
)

# Filtrage niveau commune + Rhone
df_imm17_raw = df_imm17_raw[
    (df_imm17_raw["NIVGEO"] == "COM") &
    (df_imm17_raw["CODGEO"].str.startswith(CODE_DEPT_RHONE))
].copy()

df_imm17_raw["NB"] = pd.to_numeric(df_imm17_raw["NB"], errors="coerce").fillna(0)

# Total immigres et non immigres par commune
df_imm17 = (
    df_imm17_raw
    .groupby(["CODGEO", "IMMI"])["NB"]
    .sum()
    .reset_index()
)

df_imm17 = df_imm17.pivot(index="CODGEO", columns="IMMI", values="NB").reset_index()
df_imm17.columns.name = None
df_imm17 = df_imm17.rename(columns={
    "CODGEO": "code_commune",
    "1"     : "nb_immigres_2017",
    "2"     : "nb_non_immigres_2017",
})

df_imm17["nb_immigres_2017"]     = df_imm17["nb_immigres_2017"].round(0).astype("Int64")
df_imm17["nb_non_immigres_2017"] = df_imm17["nb_non_immigres_2017"].round(0).astype("Int64")

log.info(f"Table immigration 2017 : {len(df_imm17):,} communes")

df_imm17.head(3)

2026-06-03 15:47:43 [INFO] Table immigration 2017 : 267 communes


,code_commune,nb_immigres_2017,nb_non_immigres_2017
0,69001,6,363
1,69002,4,244
2,69003,247,2633


Bloc 9-Immigration 2022 (TD_IMG1A_2022.xlsx)

Feuille COM, en-tete a la ligne 10 (index 10)
Colonnes : CODGEO | LIBGEO | AGE4XX_IMMI1_SEXE1 | AGE4XX_IMMI1_SEXE2 | ...
IMMI1 = immigres | IMMI2 = non immigres

In [52]:
df_imm22_raw = pd.read_excel(
    RAW_SOCIO / "TD_IMG1A_2022.xlsx",
    sheet_name="COM",
    engine="calamine",
    header=10,
    dtype={"CODGEO": str}
)

# Filtrage Rhone
df_imm22_raw = df_imm22_raw[
    df_imm22_raw["CODGEO"].str.startswith(CODE_DEPT_RHONE)
].copy()

# Colonnes immigres = contiennent IMMI1
# Colonnes non immigres = contiennent IMMI2
cols_immi1  = [c for c in df_imm22_raw.columns if "IMMI1" in str(c)]
cols_immi2  = [c for c in df_imm22_raw.columns if "IMMI2" in str(c)]

for col in cols_immi1 + cols_immi2:
    df_imm22_raw[col] = pd.to_numeric(df_imm22_raw[col], errors="coerce").fillna(0)

df_imm22 = pd.DataFrame()
df_imm22["code_commune"]        = df_imm22_raw["CODGEO"]
df_imm22["nb_immigres_2022"]    = df_imm22_raw[cols_immi1].sum(axis=1).round(0).astype("Int64")
df_imm22["nb_non_immigres_2022"]= df_imm22_raw[cols_immi2].sum(axis=1).round(0).astype("Int64")

log.info(f"Table immigration 2022 : {len(df_imm22):,} communes")

df_imm22.head(3)

2026-06-03 15:44:22 [INFO] Table immigration 2022 : 266 communes


,code_commune,nb_immigres_2022,nb_non_immigres_2022
26925,69001,7,390
26926,69002,4,235
26927,69003,291,2722


Bloc 10-Securite par annee : rapport qualite

In [53]:
print("Repartition securite par annee :")
print(df_securite.groupby("annee")["code_commune"].nunique().to_string())

print("\nStats total_delits par annee :")
print(df_securite.groupby("annee")["total_delits"].describe().round(2).to_string())

print("\nNaN restants :")
m = df_securite.isna().sum()
print(m[m > 0].to_string() if m.any() else "  aucune")

Repartition securite par annee :
annee
2017    275
2022    275
2024    275

Stats total_delits par annee :
       count    mean      std  min  25%   50%    75%      max
annee                                                        
2017   275.0  616.02  3579.46  0.0  0.0  20.0  138.5  54699.0
2022   275.0  672.41  3885.32  0.0  0.0  22.0  166.0  59211.0
2024   275.0  643.58  3553.09  0.0  0.0  28.0  188.5  53462.0

NaN restants :
  aucune


Bloc 11-Sauvegarde CSV

In [56]:
STAGING_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    "rhone_securite.csv"       : df_securite,
    "rhone_population.csv"     : df_pop,
    "rhone_emploi.csv"         : df_emploi,
    "rhone_revenus_2017.csv"   : df_filo17,
    "rhone_revenus_2021.csv"   : df_filo21,
    "rhone_immigration_2017.csv": df_imm17,
    "rhone_immigration_2022.csv": df_imm22,
}

for nom_fichier, df in exports.items():
    chemin = STAGING_DIR / nom_fichier
    df.to_csv(chemin, index=False, sep=";", encoding="utf-8-sig")
    log.info(f"CSV sauvegarde : {chemin}  ({len(df):,} lignes)")

2026-06-03 15:47:49 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\socioeco\rhone_securite.csv  (825 lignes)
2026-06-03 15:47:49 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\socioeco\rhone_population.csv  (275 lignes)
2026-06-03 15:47:49 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\socioeco\rhone_emploi.csv  (275 lignes)
2026-06-03 15:47:49 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\socioeco\rhone_revenus_2017.csv  (276 lignes)
2026-06-03 15:47:49 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\socioeco\rhone_revenus_2021.csv  (276 lignes)
2026-06-03 15:47:49 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\socioeco\rhone_immigration_2017.csv  (267 lignes)
2026-06-03 15:47:49 [INFO] CSV sauvegarde : C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\socioeco\rhone_immigration_2022.csv  (266 lignes)


Bloc 12-Sauvegarde SQLite

In [57]:
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
conn = sqlite3.connect(DB_PATH)

tables = {
    "securite"        : df_securite,
    "population"      : df_pop,
    "emploi"          : df_emploi,
    "revenus_2017"    : df_filo17,
    "revenus_2021"    : df_filo21,
    "immigration_2017": df_imm17,
    "immigration_2022": df_imm22,
}

for nom_table, df in tables.items():
    df_sql = df.copy()
    for col in df_sql.select_dtypes(include="Int64").columns:
        df_sql[col] = df_sql[col].astype(object).where(df_sql[col].notna(), None)
    df_sql.to_sql(nom_table, conn, if_exists="replace", index=False)
    log.info(f"Table '{nom_table}' : {len(df_sql):,} lignes inserees")

curseur = conn.cursor()
for table in tables.keys():
    curseur.execute(
        f"CREATE INDEX IF NOT EXISTS idx_{table}_com ON {table} (code_commune)"
    )

conn.commit()
conn.close()
log.info(f"Base SQLite finalisee : {DB_PATH}")

2026-06-03 15:48:24 [INFO] Table 'securite' : 825 lignes inserees
2026-06-03 15:48:24 [INFO] Table 'population' : 275 lignes inserees
2026-06-03 15:48:24 [INFO] Table 'emploi' : 275 lignes inserees
2026-06-03 15:48:24 [INFO] Table 'revenus_2017' : 276 lignes inserees
2026-06-03 15:48:24 [INFO] Table 'revenus_2021' : 276 lignes inserees
2026-06-03 15:48:24 [INFO] Table 'immigration_2017' : 267 lignes inserees
2026-06-03 15:48:24 [INFO] Table 'immigration_2022' : 266 lignes inserees
2026-06-03 15:48:24 [INFO] Base SQLite finalisee : C:\Users\NIAMBELE Siata\Desktop\MSPR2\03_database\socioeco.db


Bloc 13-Verification finale

In [58]:
from pathlib import Path
STAGING_DIR = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2\02_staging\socioeco")
DB_PATH     = Path(r"C:\Users\NIAMBELE Siata\Desktop\MSPR2\03_database\socioeco.db")

conn = sqlite3.connect(DB_PATH)
print("Tables dans la base :")
tables_presentes = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
)
for table in tables_presentes["name"]:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0, 0]
    print(f"  {table:<25} {n:>8,} lignes")
conn.close()

print("\nFichiers CSV produits :")
for f in sorted(STAGING_DIR.glob("*.csv")):
    taille = f.stat().st_size / 1024
    print(f"  {f.name:<35} {taille:>8.1f} Ko")

Tables dans la base :
  securite                       825 lignes
  population                     275 lignes
  emploi                         275 lignes
  revenus_2017                   276 lignes
  revenus_2021                   276 lignes
  immigration_2017               267 lignes
  immigration_2022               266 lignes

Fichiers CSV produits :
  rhone_emploi.csv                        19.4 Ko
  rhone_immigration_2017.csv               4.0 Ko
  rhone_immigration_2022.csv               4.0 Ko
  rhone_population.csv                    19.9 Ko
  rhone_revenus_2017.csv                  13.3 Ko
  rhone_revenus_2021.csv                   7.7 Ko
  rhone_securite.csv                      25.1 Ko
